In [6]:
import polars as pl
import gc
import glob
import os

In [3]:
target_path = "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\*.csv"
file_paths = glob.glob(target_path)
print(file_paths)
# file_name = file_paths[0].split('/')[-1]
# print(file_name)

['C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\dataset1.csv', 'C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\dataset10.csv', 'C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\dataset11.csv', 'C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\dataset12.csv', 'C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\dataset13.csv', 'C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\dataset14.csv', 'C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\dataset15.csv', 'C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\dataset16.csv', 'C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\dataset17.csv', 'C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\dataset18.csv', 'C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\dataset19.csv', 'C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\dataset2

In [35]:
target_path = "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\archive\\*.csv"
file_paths = glob.glob(target_path)

print(f"Found {len(file_paths)} datasets. Running surgical audit...\n")
total_malicious = 0
# Loop through each file, but ONLY scan the final messy column
for file in sorted(file_paths):
    file_name = file.split('\\')[-1]
    
    try:
        # Lazy scan, selecting only the column we need to parse
        lf = pl.scan_csv(
            file,
            ignore_errors=True,
            null_values=["-", "(empty)"]
        ).select([
            pl.col("tunnel_parents   label   detailed-label").str.split("   ").list.get(1).alias("label")
        ])
        
        # Trigger the execution for this specific file
        counts = lf.group_by("label").count().collect()
        
        # Extract numbers safely
        labels_present = counts["label"].to_list()
        benign_count = counts.filter(pl.col("label") == "Benign")["count"][0] if "Benign" in labels_present else 0
        malicious_count = counts.filter(pl.col("label") != "Benign")["count"].sum()
        
        print(f"{file_name:<15} -> Benign: {benign_count:<10} | Malicious: {malicious_count:<10}")
        
        total_malicious += malicious_count

    except Exception as e:
        print(f"{file_name:<15} -> ERROR reading file: {e}")

print(f"\nTotal Malicious Flows Across All Files: {total_malicious}")

Found 23 datasets. Running surgical audit...

dataset1.csv    -> Benign: 0          | Malicious: 452       


C:\Users\babai\AppData\Local\Temp\ipykernel_10860\389290745.py:21: DeprecationWarning: `count` was renamed; use `len` instead
  counts = lf.group_by("label").count().collect()


dataset10.csv   -> Benign: 3734       | Malicious: 3390604   
dataset11.csv   -> Benign: 211        | Malicious: 26        
dataset12.csv   -> Benign: 20574934   | Malicious: 46746875  
dataset13.csv   -> Benign: 4420       | Malicious: 6         
dataset14.csv   -> Benign: 7337       | Malicious: 73561644  
dataset15.csv   -> Benign: 2663       | Malicious: 13642435  
dataset16.csv   -> Benign: 8262389    | Malicious: 2185398   
dataset17.csv   -> Benign: 1923       | Malicious: 21222     
dataset18.csv   -> Benign: 1380791    | Malicious: 53073800  
dataset19.csv   -> Benign: 4536       | Malicious: 151567    
dataset2.csv    -> Benign: 0          | Malicious: 1374      
dataset20.csv   -> Benign: 3272       | Malicious: 14        
dataset21.csv   -> Benign: 3193       | Malicious: 16        
dataset22.csv   -> Benign: 31438      | Malicious: 54628417  
dataset23.csv   -> Benign: 469275     | Malicious: 539473    
dataset3.csv    -> Benign: 0          | Malicious: 130       
dataset4

In [36]:
import os

#borrowed schema from https://www.kaggle.com/code/harishragavender0209/iot23-peoject
IOT23_SCHEMA = {
    'ts': pl.Float64,              # Timestamp needs precision
    'uid': pl.String,              # Connection unique ID
    'id.orig_h': pl.String,        # Source IP
    'id.orig_p': pl.UInt32,        # Source Port (Unsigned Int, ports can't be negative)
    'id.resp_h': pl.String,        # Destination IP
    'id.resp_p': pl.UInt32,        # Destination Port
    'proto': pl.Categorical,       # Protocol (tcp, udp, icmp) - Categorical saves massive RAM
    'service': pl.String,          # http, dns, ssh, etc. 
    'duration': pl.Float32,        # Downcast to Float32
    'orig_bytes': pl.String,       # Read as string first because of '-' nulls
    'resp_bytes': pl.String,       
    'conn_state': pl.Categorical,  # S0, SF, REJ, etc.
    'local_orig': pl.String,       
    'local_resp': pl.String,       
    'missed_bytes': pl.Float32,    
    'history': pl.String,          
    'orig_pkts': pl.Float32,       
    'orig_ip_bytes': pl.Float32,   
    'resp_pkts': pl.Float32,       
    'resp_ip_bytes': pl.Float32,   
    # The messy column as a single string first
    'tunnel_parents   label   detailed-label': pl.String 
}

def parse_and_clean_schema(file_path, file_name, chunk_path, typ):
    lf = pl.scan_csv(
        file_path,
        schema=IOT23_SCHEMA,
        ignore_errors=True,
        null_values=["-", "(empty)"]
    )

    lf_cleaned = lf.with_columns([
        pl.col("orig_bytes").cast(pl.Float32),
        pl.col("resp_bytes").cast(pl.Float32),
        pl.col("tunnel_parents   label   detailed-label").str.split(by="   ").alias("split_target")
        #splitting the messy column into three separate columns
    ]).with_columns([pl.col("split_target").list.get(0).alias("tunnel_parents"),
    pl.col("split_target").list.get(1).alias("label").cast(pl.Categorical),
    pl.col("split_target").list.get(2).alias("detailed-label").cast(pl.Categorical)]
    #categorical for memory efficiency
    ).drop(["tunnel_parents   label   detailed-label", "split_target"])

    if typ == "benign":
        lf_benign = lf_cleaned.filter(pl.col("label").is_in(["Benign", "benign"]))
    elif typ == "malicious":
        lf_benign = lf_cleaned.filter(~pl.col("label").is_in(["Benign", "benign"]))
    df_benign = lf_benign.collect()

    if df_benign.height > 0:
        df_benign.write_parquet(chunk_path)
        print(f"Saved {df_benign.height} {typ} rows from {file_name}.")
    else:
        print(f"Skipped {file_name} (0 Benign rows).")

    del df_benign
    gc.collect()


In [28]:
temp_dir = "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\benign_chunks"
os.makedirs(temp_dir, exist_ok=True)

for file in sorted(file_paths):
    file_name = os.path.basename(file)
    chunk_path = os.path.join(temp_dir, f"benign_{file_name.replace('.csv', '.parquet')}")

    parse_and_clean_schema(file, file_name, chunk_path, typ="benign")

final_benign_path = "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\benign_master.parquet"
pl.scan_parquet(f"{temp_dir}/*.parquet").sink_parquet(final_benign_path)

print("successfully merged all benign chunks into a master parquet file.")

Saved 452 benign rows from dataset1.csv.
Saved 3734 benign rows from dataset10.csv.
Saved 211 benign rows from dataset11.csv.
Saved 20574934 benign rows from dataset12.csv.
Saved 4420 benign rows from dataset13.csv.
Saved 7337 benign rows from dataset14.csv.
Saved 2663 benign rows from dataset15.csv.
Saved 8262389 benign rows from dataset16.csv.
Saved 1923 benign rows from dataset17.csv.
Saved 1380791 benign rows from dataset18.csv.
Saved 4536 benign rows from dataset19.csv.
Saved 1374 benign rows from dataset2.csv.
Saved 3272 benign rows from dataset20.csv.
Saved 3193 benign rows from dataset21.csv.
Saved 31438 benign rows from dataset22.csv.
Saved 469275 benign rows from dataset23.csv.
Saved 130 benign rows from dataset3.csv.
Saved 22548 benign rows from dataset4.csv.
Saved 2181 benign rows from dataset5.csv.
Saved 75955 benign rows from dataset6.csv.
Saved 2476 benign rows from dataset7.csv.
Saved 1794 benign rows from dataset8.csv.
Saved 3665 benign rows from dataset9.csv.
successf

In [6]:
final_benign_path = "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\benign_master.parquet" #redecalared for block running
total_benign = pl.scan_parquet(final_benign_path).select(pl.len()).collect().item()
target_malicious = total_benign * 3
print(f"Total Benign Flows: {total_benign}")
print(f"Target Malicious Flows (1:3): {target_malicious}")

total_malicious = 294451211 
sample_fraction = target_malicious / total_malicious
hash_threshold = int(sample_fraction * 100000)

temp_dir = "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\malicious_chunks"
os.makedirs(temp_dir, exist_ok=True)

for file in sorted(file_paths):
    file_name = os.path.basename(file)
    chunk_path = os.path.join(temp_dir, f"malicious_{file_name.replace('.csv', '.parquet')}")

    parse_and_clean_schema(file, file_name, chunk_path, typ="malicious")

final_malicious_path = "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\malicious_master.parquet"
pl.scan_parquet(f"{temp_dir}/*.parquet").sink_parquet(final_malicious_path)

print("successfully merged all malicious chunks into a master parquet file.")

Total Benign Flows: 30860691
Target Malicious Flows (1:3): 92582073
Skipped dataset1.csv (0 Benign rows).
Saved 3390604 malicious rows from dataset10.csv.
Saved 26 malicious rows from dataset11.csv.
Saved 46746875 malicious rows from dataset12.csv.
Saved 6 malicious rows from dataset13.csv.
Saved 73561644 malicious rows from dataset14.csv.
Saved 13642435 malicious rows from dataset15.csv.
Saved 2185398 malicious rows from dataset16.csv.
Saved 21222 malicious rows from dataset17.csv.
Saved 53073800 malicious rows from dataset18.csv.
Saved 151567 malicious rows from dataset19.csv.
Skipped dataset2.csv (0 Benign rows).
Saved 14 malicious rows from dataset20.csv.
Saved 16 malicious rows from dataset21.csv.
Saved 54628417 malicious rows from dataset22.csv.
Saved 539473 malicious rows from dataset23.csv.
Skipped dataset3.csv (0 Benign rows).
Saved 6355745 malicious rows from dataset4.csv.
Saved 8222 malicious rows from dataset5.csv.
Saved 11378759 malicious rows from dataset6.csv.
Saved 3578

In [ ]:
def clean_and_impute(input_path, output_path):
    
    lf = pl.scan_parquet(input_path)
    existing_cols = lf.collect_schema().names()

    cols_to_drop = ["uid", "local_orig", "local_resp"]
    actual_drops = [col for col in cols_to_drop if col in existing_cols]
    lf = lf.drop(actual_drops)

    num_cols = [
        "duration", "orig_bytes", "resp_bytes", "missed_bytes", 
        "orig_pkts", "orig_ip_bytes", "resp_pkts", "resp_ip_bytes"
    ]
    cat_cols = ["service", "history", "conn_state", "proto", "label", "detailed-label"]

    num_imputes = [pl.col(c).fill_null(0.0) for c in num_cols if c in existing_cols]
    cat_imputes = [pl.col(c).fill_null("unknown") for c in cat_cols if c in existing_cols]

    lf_cleaned = lf.with_columns(num_imputes + cat_imputes)

    print(f"Streaming cleaned data to {output_path}...")
    lf_cleaned.sink_parquet(output_path)
    
    print(f"SUCCESS: Cleaned dataset saved -> {output_path}\n")

In [3]:
clean_and_impute("C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\benign_master.parquet", "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\benign_cleaned.parquet")
gc.collect()

clean_and_impute("C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\malicious_master.parquet", "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\malicious_cleaned.parquet")
gc.collect()

Streaming cleaned data to C:\Users\babai\OneDrive\Desktop\CaseStudiesDatasets\benign_cleaned.parquet...
SUCCESS: Cleaned dataset saved -> C:\Users\babai\OneDrive\Desktop\CaseStudiesDatasets\benign_cleaned.parquet

Streaming cleaned data to C:\Users\babai\OneDrive\Desktop\CaseStudiesDatasets\malicious_cleaned.parquet...
SUCCESS: Cleaned dataset saved -> C:\Users\babai\OneDrive\Desktop\CaseStudiesDatasets\malicious_cleaned.parquet



0

In [37]:
def fuse_and_structure():
    
    lf_benign = pl.scan_parquet("C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\benign_cleaned.parquet")
    lf_malicious = pl.scan_parquet("C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\malicious_cleaned.parquet")
    
    lf_combined = pl.concat([lf_benign, lf_malicious])
    
    selected_columns = [
        "ts", "id.orig_h",             # MUST keep for sorting and temporal splits
        "id.orig_p", "id.resp_p", "proto", "duration", 
        "orig_bytes", "resp_bytes", "conn_state", 
        "history", "orig_pkts", "resp_pkts",
        "label", "detailed-label" # Targets
    ]
    
    print("Applying chronological sort (this is computationally heavy)...")
    lf_final = (
        lf_combined
        .select(selected_columns)
        .sort(["id.orig_h", "ts"]) 
    )
    
    output_path = "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\iot23_master_chronological_v2.parquet"
    lf_final.sink_parquet(output_path)
    
    print(f"SUCCESS: Master chronological dataset saved to {output_path}")

In [38]:
fuse_and_structure()
gc.collect()

Applying chronological sort (this is computationally heavy)...
SUCCESS: Master chronological dataset saved to C:\Users\babai\OneDrive\Desktop\CaseStudiesDatasets\iot23_master_chronological_v2.parquet


5091

In [7]:
def split_by_ip(selected_ips: list[str]):
    
    master_path = "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\iot23_master_chronological_v2.parquet"
    output_dir = "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\per_device\\"
    
    os.makedirs(output_dir, exist_ok=True)
    
    lf = pl.scan_parquet(master_path)
    
    for ip in selected_ips:
        print(f"Processing {ip}...")
        
        # Sanitize IP for use as filename
        safe_ip = ip.replace(":", "_").replace(".", "_")
        out_path = os.path.join(output_dir, f"device_{safe_ip}.parquet")
        
        (
            lf
            .filter(pl.col("id.orig_h") == ip)
            # Already sorted chronologically from fuse_and_structure, but re-sort per device to be safe
            .sort("ts")
            .sink_parquet(out_path)
        )
        
        print(f"  Saved → {out_path}")
    
    print(f"\nDone. {len(selected_ips)} device file(s) written to {output_dir}")

In [8]:
split_by_ip([
    "192.168.1.198",
    "192.168.1.197",
    "192.168.1.194",
    "192.168.100.111",
    "192.168.100.108",
    "192.168.1.196",
    "192.168.1.193",
    "192.168.1.195",
    "192.168.1.200",
    "192.168.100.103",
    "192.168.2.5",
    "196.118.25.105",
    "192.168.100.113",
    "162.248.88.215",
    "192.168.100.1",
    "192.168.2.3",
    "192.168.1.132",
    "197.13.3.22",
    "156.62.3.247",
    "192.168.1.199",
    "176.32.33.171",
    "0.0.0.0",
    "212.191.225.6",
    "4.68.110.10",
    "194.70.98.42",
    "41.208.48.126",
    "196.192.102.9",
    "fe80::5bcc:698e:39d5:cdf",
    "197.218.1.145",
    "192.168.2.1",
    "165.0.38.126",
    "196.207.210.182",
    "192.168.1.1",
    "169.254.15.115",
])
gc.collect()

Processing 192.168.1.198...
  Saved → C:\Users\babai\OneDrive\Desktop\CaseStudiesDatasets\per_device\device_192_168_1_198.parquet
Processing 192.168.1.197...
  Saved → C:\Users\babai\OneDrive\Desktop\CaseStudiesDatasets\per_device\device_192_168_1_197.parquet
Processing 192.168.1.194...
  Saved → C:\Users\babai\OneDrive\Desktop\CaseStudiesDatasets\per_device\device_192_168_1_194.parquet
Processing 192.168.100.111...
  Saved → C:\Users\babai\OneDrive\Desktop\CaseStudiesDatasets\per_device\device_192_168_100_111.parquet
Processing 192.168.100.108...
  Saved → C:\Users\babai\OneDrive\Desktop\CaseStudiesDatasets\per_device\device_192_168_100_108.parquet
Processing 192.168.1.196...
  Saved → C:\Users\babai\OneDrive\Desktop\CaseStudiesDatasets\per_device\device_192_168_1_196.parquet
Processing 192.168.1.193...
  Saved → C:\Users\babai\OneDrive\Desktop\CaseStudiesDatasets\per_device\device_192_168_1_193.parquet
Processing 192.168.1.195...
  Saved → C:\Users\babai\OneDrive\Desktop\CaseStudiesD

1143

In [ ]:
def inspect_all_devices(all_ips: list[str]):

    for ip in all_ips:
        safe_ip = ip.replace(":", "_").replace(".", "_")
        path = f"C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\per_device\\device_{safe_ip}.parquet"

        lf = pl.scan_parquet(path)

        # ── 1. Shape & Schema ──────────────────────────────────────────
        row_count = lf.select(pl.len()).collect().item()
        schema = lf.schema

        print("=" * 55)
        print(f"DEVICE: {ip}")
        print("=" * 55)
        print(f"\n📐 Shape: {row_count:,} rows × {len(schema)} columns")
        print(f"\n🔢 Schema:\n{schema}")

        # ── 2. Null Counts ─────────────────────────────────────────────
        print("\n🕳️  Null Counts:")
        print(lf.select(pl.all().is_null().sum()).collect())

        # ── 3. Chronological Order Check ──────────────────────────────
        print("\n🕐 Chronological Order Check:")
        ts_stats = lf.select([
            pl.col("ts").min().alias("ts_min"),
            pl.col("ts").max().alias("ts_max"),
            (pl.col("ts").max() - pl.col("ts").min()).alias("ts_range_sec"),
            (pl.col("ts").diff().drop_nulls() >= 0).all().alias("is_sorted")
        ]).collect()
        print(ts_stats)

        # ── 4. Label Split ────────────────────────────────────────────
        print("\n🏷️  Label Split:")
        print(lf.group_by("label").agg(pl.len().alias("count")).sort("count", descending=True).collect())

        print("\n🏷️  Detailed-Label Split:")
        print(lf.group_by("detailed-label").agg(pl.len().alias("count")).sort("count", descending=True).collect())

        print("\n")


NameError: name 'all_ips' is not defined

In [12]:
all_ips = [
    "192.168.1.198",
    "192.168.1.197",
    "192.168.1.194",
    "192.168.100.111",
    "192.168.100.108",
    "192.168.1.196",
    "192.168.1.193",
    "192.168.1.195",
    "192.168.1.200",
    "192.168.100.103",
    "192.168.2.5",
    "196.118.25.105",
    "192.168.100.113",
    "162.248.88.215",
    "192.168.100.1",
    "192.168.2.3",
    "192.168.1.132",
    "197.13.3.22",
    "156.62.3.247",
    "192.168.1.199",
    "176.32.33.171",
    "0.0.0.0",
    "212.191.225.6",
    "4.68.110.10",
    "194.70.98.42",
    "41.208.48.126",
    "196.192.102.9",
    "fe80::5bcc:698e:39d5:cdf",
    "197.218.1.145",
    "192.168.2.1",
    "165.0.38.126",
    "196.207.210.182",
    "192.168.1.1",
    "169.254.15.115",
]

In [13]:

inspect_all_devices(all_ips)

C:\Users\babai\AppData\Local\Temp\ipykernel_18416\1149930298.py:11: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  schema = lf.schema


DEVICE: 192.168.1.198

📐 Shape: 80,965,996 rows × 14 columns

🔢 Schema:
Schema({'ts': Float64, 'id.orig_h': String, 'id.orig_p': UInt32, 'id.resp_p': UInt32, 'proto': Categorical, 'duration': Float32, 'orig_bytes': Float32, 'resp_bytes': Float32, 'conn_state': Categorical, 'history': String, 'orig_pkts': Float32, 'resp_pkts': Float32, 'label': Categorical, 'detailed-label': Categorical})

🕳️  Null Counts:
shape: (1, 14)
┌─────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────┬────────────────┐
│ ts  ┆ id.orig_h ┆ id.orig_p ┆ id.resp_p ┆ … ┆ orig_pkts ┆ resp_pkts ┆ label ┆ detailed-label │
│ --- ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---   ┆ ---            │
│ u32 ┆ u32       ┆ u32       ┆ u32       ┆   ┆ u32       ┆ u32       ┆ u32   ┆ u32            │
╞═════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════╪════════════════╡
│ 0   ┆ 0         ┆ 0         ┆ 0         ┆ … ┆ 0         ┆ 0         ┆ 0     ┆ 0          

In [ ]:
valid_ips = ["192_168_100_113","192_168_2_5","192_168_100_103","192_168_1_200","192_168_1_195","192_168_1_193","192_168_1_196","192_168_100_108","192_168_100_111","192_168_1_194","192_168_1_197","192_168_1_198"]

In [16]:
with pl.Config(tbl_rows=100, tbl_cols=20, fmt_str_lengths=100, tbl_width_chars=500):
    inspect_all_devices(valid_ips)

C:\Users\babai\AppData\Local\Temp\ipykernel_18416\1149930298.py:11: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  schema = lf.schema


DEVICE: 192_168_100_113

📐 Shape: 13,685 rows × 14 columns

🔢 Schema:
Schema({'ts': Float64, 'id.orig_h': String, 'id.orig_p': UInt32, 'id.resp_p': UInt32, 'proto': Categorical, 'duration': Float32, 'orig_bytes': Float32, 'resp_bytes': Float32, 'conn_state': Categorical, 'history': String, 'orig_pkts': Float32, 'resp_pkts': Float32, 'label': Categorical, 'detailed-label': Categorical})

🕳️  Null Counts:
shape: (1, 14)
┌─────┬───────────┬───────────┬───────────┬───────┬──────────┬────────────┬────────────┬────────────┬─────────┬───────────┬───────────┬───────┬────────────────┐
│ ts  ┆ id.orig_h ┆ id.orig_p ┆ id.resp_p ┆ proto ┆ duration ┆ orig_bytes ┆ resp_bytes ┆ conn_state ┆ history ┆ orig_pkts ┆ resp_pkts ┆ label ┆ detailed-label │
│ --- ┆ ---       ┆ ---       ┆ ---       ┆ ---   ┆ ---      ┆ ---        ┆ ---        ┆ ---        ┆ ---     ┆ ---       ┆ ---       ┆ ---   ┆ ---            │
│ u32 ┆ u32       ┆ u32       ┆ u32       ┆ u32   ┆ u32      ┆ u32        ┆ u32        ┆ u32   

In [21]:
def head_all_devices(all_ips: list[str], n: int = 100):
    
    for ip in all_ips:
        safe_ip = ip.replace(":", "_").replace(".", "_")
        path = f"C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\per_device\\device_{safe_ip}.parquet"
        
        print("=" * 55)
        print(f"DEVICE: {ip}")
        print("=" * 55)
        
        print(
            pl.scan_parquet(path)
            .head(n)
            .collect()
        )
        print("\n")


In [22]:
with pl.Config(tbl_rows=100, tbl_cols=20, fmt_str_lengths=100, tbl_width_chars=500):
    head_all_devices(valid_ips)

DEVICE: 192_168_100_113
shape: (100, 14)
┌──────────┬─────────────────┬───────────┬───────────┬───────┬──────────┬────────────┬────────────┬────────────┬─────────┬───────────┬───────────┬────────┬────────────────┐
│ ts       ┆ id.orig_h       ┆ id.orig_p ┆ id.resp_p ┆ proto ┆ duration ┆ orig_bytes ┆ resp_bytes ┆ conn_state ┆ history ┆ orig_pkts ┆ resp_pkts ┆ label  ┆ detailed-label │
│ ---      ┆ ---             ┆ ---       ┆ ---       ┆ ---   ┆ ---      ┆ ---        ┆ ---        ┆ ---        ┆ ---     ┆ ---       ┆ ---       ┆ ---    ┆ ---            │
│ f64      ┆ str             ┆ u32       ┆ u32       ┆ cat   ┆ f32      ┆ f32        ┆ f32        ┆ cat        ┆ str     ┆ f32       ┆ f32       ┆ cat    ┆ cat            │
╞══════════╪═════════════════╪═══════════╪═══════════╪═══════╪══════════╪════════════╪════════════╪════════════╪═════════╪═══════════╪═══════════╪════════╪════════════════╡
│ 1.5330e9 ┆ 192.168.100.113 ┆ 123       ┆ 123       ┆ udp   ┆ 0.00549  ┆ 48.0       ┆ 48.0   

In [39]:
import pandas as pd
import pyarrow.parquet as pq

PATH = "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\iot23_master_chronological_v2.parquet"
pf = pq.ParquetFile(PATH)
first_batch = next(pf.iter_batches(batch_size=500_000))
df = first_batch.to_pandas()

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(df.dtypes)

Loaded: 122,880 rows × 14 columns
ts                 float64
id.orig_h           object
id.orig_p           uint32
id.resp_p           uint32
proto             category
duration           float32
orig_bytes         float32
resp_bytes         float32
conn_state        category
history             object
orig_pkts          float32
resp_pkts          float32
label             category
detailed-label    category
dtype: object


In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve
from torch.utils.data import DataLoader, IterableDataset

In [2]:
PATH       = "C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\iot23_master_chronological_v2.parquet"  
CAT_COLS   = ['proto', 'conn_state']
DROP_COLS  = ['ts', 'history', 'detailed-label', 'label']
SEQ_LEN    = 32
HIDDEN_DIM = 128
LATENT_DIM = 16
BATCH_SIZE = 256
EPOCHS     = 10
LMBDA      = 0.5
CHUNK_SIZE = 10_000
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {DEVICE}")


Using device: cuda


In [6]:
import polars as pl

lf = pl.scan_parquet(PATH)

device_counts = (
    lf.group_by("id.orig_h")
    .agg(pl.len().alias("n"))
    .collect(streaming=True)
    .sort("n", descending=True)
)

print(device_counts.head(10))          # top 10 busiest devices
print(f"\nDevices with >= 51 rows: {(device_counts['n'] >= 51).sum():,}")
print(f"Devices with <  51 rows: {(device_counts['n'] <  51).sum():,}")

C:\Users\babai\AppData\Local\Temp\ipykernel_18820\2143765142.py:8: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True)


shape: (10, 2)
┌─────────────────┬──────────┐
│ id.orig_h       ┆ n        │
│ ---             ┆ ---      │
│ str             ┆ u32      │
╞═════════════════╪══════════╡
│ 192.168.1.198   ┆ 80965996 │
│ 192.168.1.197   ┆ 74229331 │
│ 192.168.1.194   ┆ 73567464 │
│ 192.168.100.111 ┆ 61021705 │
│ 192.168.100.108 ┆ 11381812 │
│ 192.168.1.196   ┆ 10446132 │
│ 192.168.1.193   ┆ 5408909  │
│ 192.168.1.195   ┆ 3602157  │
│ 192.168.1.200   ┆ 3392003  │
│ 192.168.100.103 ┆ 994268   │
└─────────────────┴──────────┘

Devices with >= 51 rows: 34
Devices with <  51 rows: 49,451


In [10]:
import polars as pl


viable_devices = (
    pl.scan_parquet(PATH)
    .group_by("id.orig_h")
    .agg(pl.len().alias("n"))
    .filter(pl.col("n") >= 51)
    .sort("n", descending=True)
    .collect(streaming=True)
)

with pl.Config(tbl_rows=100):
    print(viable_devices)
print(f"\nTotal viable devices: {len(viable_devices)}")

C:\Users\babai\AppData\Local\Temp\ipykernel_18820\2684541098.py:10: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True)


shape: (34, 2)
┌──────────────────────────┬──────────┐
│ id.orig_h                ┆ n        │
│ ---                      ┆ ---      │
│ str                      ┆ u32      │
╞══════════════════════════╪══════════╡
│ 192.168.1.198            ┆ 80965996 │
│ 192.168.1.197            ┆ 74229331 │
│ 192.168.1.194            ┆ 73567464 │
│ 192.168.100.111          ┆ 61021705 │
│ 192.168.100.108          ┆ 11381812 │
│ 192.168.1.196            ┆ 10446132 │
│ 192.168.1.193            ┆ 5408909  │
│ 192.168.1.195            ┆ 3602157  │
│ 192.168.1.200            ┆ 3392003  │
│ 192.168.100.103          ┆ 994268   │
│ 192.168.2.5              ┆ 154886   │
│ 196.118.25.105           ┆ 65726    │
│ 192.168.100.113          ┆ 13685    │
│ 162.248.88.215           ┆ 2013     │
│ 192.168.100.1            ┆ 1653     │
│ 192.168.2.3              ┆ 979      │
│ 192.168.1.132            ┆ 452      │
│ 197.13.3.22              ┆ 357      │
│ 156.62.3.247             ┆ 273      │
│ 192.168.1.199          

In [3]:
def make_sequences(path, window=50, sample_n=100_000):

    lf = pl.scan_parquet(path)

    # Count rows per device
    device_counts = (
        lf.group_by("id.orig_h")
        .agg(pl.len().alias("n"))
        .collect(streaming=True)
    )
    total = device_counts["n"].sum()

    # Sample proportionally from each device
    chunks = []
    for row in device_counts.iter_rows(named=True):
        device_sample = max(window + 1, int(sample_n * row["n"] / total))
        chunk = (
            lf.filter(pl.col("id.orig_h") == row["id.orig_h"])
            .collect(streaming=True)
            .sample(n=min(device_sample, row["n"]), shuffle=True)
        )
        chunks.append(chunk)

    df = pl.concat(chunks).to_pandas()
    print(f"Loaded {len(df):,} rows across {len(chunks)} devices")
    # df = pl.read_parquet(path).to_pandas()

    cat_cols = ["proto", "conn_state", "history"]
    for col in cat_cols:
        if col in df.columns:
            df[col] = LabelEncoder().fit_transform(df[col].astype(str))

    feature_cols = [
        "id.orig_p", "id.resp_p", "proto", "duration",
        "orig_bytes", "resp_bytes", "conn_state",
        "history", "orig_pkts", "resp_pkts"
    ]
    label_col = "label"

    df[label_col] = LabelEncoder().fit_transform(df[label_col].astype(str))

    scaler = StandardScaler()
    df[feature_cols] = scaler.fit_transform(df[feature_cols].fillna(0))

    # ── Pre-calculate total sequences so we can allocate once ────────────────
    groups = {ip: g.sort_values("ts").reset_index(drop=True)
              for ip, g in df.groupby("id.orig_h", sort=False)
              if len(g) >= window + 1}

    total_seqs = sum(len(g) - window for g in groups.values())
    F = len(feature_cols)

    # ── Allocate output arrays upfront — no list appending ──────────────────
    X = np.empty((total_seqs, window, F), dtype=np.float32)
    y = np.empty(total_seqs,              dtype=np.int64)

    idx = 0
    for device_ip, group in groups.items():
        vals   = group[feature_cols].values.astype(np.float32)
        labels = group[label_col].values
        n      = len(vals) - window

        for i in range(n):
            X[idx] = vals[i : i + window]
            y[idx] = labels[i + window]
            idx += 1

    print(f"Sequences: {X.shape}  |  Labels: {y.shape}")
    return X, y

In [4]:
import polars as pl
df = pl.scan_parquet(PATH).select(pl.len()).collect().item()
print(f"Rows in file: {df:,}")

Rows in file: 325,309,946


In [5]:
X, y = make_sequences(PATH, window=50)

C:\Users\babai\AppData\Local\Temp\ipykernel_18820\1223297406.py:9: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True)
C:\Users\babai\AppData\Local\Temp\ipykernel_18820\1223297406.py:19: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True)


Loaded 159,859 rows across 49485 devices
Sequences: (99427, 50, 10)  |  Labels: (99427,)


In [30]:
def preprocess_chunk(df, scaler, feature_cols, num_cols, ohe_cols):
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors='ignore')
    for col in df.select_dtypes(['category']).columns:
        df[col] = df[col].astype(str)
    df = df.fillna(0)
    df = pd.get_dummies(df, columns=[c for c in CAT_COLS if c in df.columns])
    for col in feature_cols:
        if col not in df.columns:
            df[col] = 0
    df = df[feature_cols]
    # clip and scale numeric only
    df[num_cols] = df[num_cols].clip(
        lower=df[num_cols].quantile(0.01),
        upper=df[num_cols].quantile(0.99),
        axis=1
    )
    df[num_cols] = scaler.transform(df[num_cols].values)
    # ohe cols stay as 0/1 untouched
    return df


def fit_scaler_and_cols(path, sample_rows=100_000):
    pf     = pq.ParquetFile(path)
    frames = []
    rows   = 0
    for batch in pf.iter_batches(batch_size=CHUNK_SIZE):
        df = batch.to_pandas()
        df = df[df['label'] == 'Benign']
        if df.empty:
            continue
        df = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors='ignore')
        for col in df.select_dtypes(['category']).columns:
            df[col] = df[col].astype(str)
        df = df.fillna(0)
        df = pd.get_dummies(df, columns=[c for c in CAT_COLS if c in df.columns])
        frames.append(df)
        rows += len(df)
        if rows >= sample_rows:
            break

    sample       = pd.concat(frames, ignore_index=True).fillna(0)
    feature_cols = sample.columns.tolist()

     # identify which columns are one-hot vs numeric
    ohe_cols     = [c for c in feature_cols if any(c.startswith(cat + '_') for cat in CAT_COLS)]
    num_cols     = [c for c in feature_cols if c not in ohe_cols]

    # clip outliers on numeric only
    sample[num_cols] = sample[num_cols].clip(
        lower=sample[num_cols].quantile(0.01),
        upper=sample[num_cols].quantile(0.99),
        axis=1
    )

    # fit scaler on numeric columns only
    scaler = StandardScaler()
    scaler.fit(sample[num_cols].values)

    print(f"Numeric cols scaled: {num_cols}")
    print(f"OHE cols left as 0/1: {ohe_cols}")
    print(f"Total features: {len(feature_cols)}")
    print(f"Benign sample rows: {len(sample):,}")
    # scaler       = StandardScaler()
    # scaler.fit(sample.values)
    # print(f"Features after encoding: {len(feature_cols)}")
    # print(f"Benign sample rows for scaler: {len(sample):,}")
    return scaler, feature_cols, num_cols, ohe_cols

In [26]:
class ParquetSequenceDataset(IterableDataset):
    def __init__(self, path, scaler, feature_cols, num_cols, ohe_cols, benign_only=False, yield_labels=False, max_sequences=None):
        self.path         = path
        self.scaler       = scaler
        self.feature_cols = feature_cols
        self.benign_only  = benign_only
        self.yield_labels = yield_labels
        self.max_sequences  = max_sequences
        self.num_cols     = num_cols
        self.ohe_cols     = ohe_cols

    def __iter__(self):
        pf      = pq.ParquetFile(self.path)
        buf_x   = []
        buf_lbl = []

        for batch in pf.iter_batches(batch_size=CHUNK_SIZE):
            df     = batch.to_pandas()
            labels = (df['label'] != 'Benign').astype(int).values

            if self.benign_only:
                mask   = df['label'] == 'Benign'
                df     = df[mask]
                labels = labels[mask.values]
                if df.empty:
                    continue

            df = preprocess_chunk(df, self.scaler, self.feature_cols,
                                  self.num_cols, self.ohe_cols) 
            X  = df.values.astype(np.float32)

            for row, lbl in zip(X, labels):
                if lbl == 1:
                    # malicious row — yield it as its own single-row sequence
                    # and reset the buffer so it never contaminates a benign window
                    buf_x   = []
                    buf_lbl = []
                    if self.yield_labels:
                        yield torch.tensor(row).unsqueeze(0).expand(SEQ_LEN, -1), torch.tensor(1)
                else:
                    buf_x.append(row)
                    buf_lbl.append(lbl)

                while len(buf_x) >= SEQ_LEN:
                    seq = np.stack(buf_x[:SEQ_LEN])
                    lbl_seq = 0  # always benign since we reset on malicious
                    if self.yield_labels:
                        yield torch.tensor(seq), torch.tensor(lbl_seq)
                    else:
                        yield torch.tensor(seq)
                    buf_x   = buf_x[SEQ_LEN:]
                    buf_lbl = buf_lbl[SEQ_LEN:]

In [32]:
import itertools

scaler, feature_cols, num_cols, ohe_cols = fit_scaler_and_cols(PATH, sample_rows=50_000)

print("Numeric cols (will be scaled):")
print(num_cols)

print("\nOHE cols (will stay 0/1):")
print(ohe_cols)

# df = preprocess_chunk(df, scaler, feature_cols, num_cols, ohe_cols)

dataset = ParquetSequenceDataset(PATH, scaler, feature_cols,num_cols, ohe_cols,
                                  benign_only=True,
                                  yield_labels=True)

# hard cap with islice — guaranteed to stop after 3
for i, (seq, lbl) in enumerate(itertools.islice(dataset, 3)):
    print(f"\n{'='*60}")
    print(f"Sequence {i+1} | label: {'MALICIOUS' if lbl.item() == 1 else 'BENIGN'}")
    print(f"Shape: {seq.shape}")
    print(f"Stats — min: {seq.min():.3f} max: {seq.max():.3f} mean: {seq.mean():.3f}")
    print(f"\nFirst 3 timesteps (rows):")
    for t in range(min(3, seq.shape[0])):
        print(f"  t={t}: {seq[t].numpy().round(3)}")
    print(f"\nLast timestep:")
    print(f"  t={seq.shape[0]-1}: {seq[-1].numpy().round(3)}")
    print(f"\nPer-feature means across sequence:")
    means = seq.mean(dim=0).numpy().round(3)
    for col, val in zip(feature_cols, means):
        print(f"  {col:30s}: {val:.3f}")

Numeric cols scaled: ['id.orig_p', 'id.resp_p', 'duration', 'orig_bytes', 'resp_bytes', 'orig_pkts', 'resp_pkts']
OHE cols left as 0/1: ['proto_icmp', 'proto_tcp', 'proto_udp', 'conn_state_OTH', 'conn_state_REJ', 'conn_state_RSTOS0', 'conn_state_S0', 'conn_state_SF', 'conn_state_RSTR', 'conn_state_S1']
Total features: 17
Benign sample rows: 54,132
Numeric cols (will be scaled):
['id.orig_p', 'id.resp_p', 'duration', 'orig_bytes', 'resp_bytes', 'orig_pkts', 'resp_pkts']

OHE cols (will stay 0/1):
['proto_icmp', 'proto_tcp', 'proto_udp', 'conn_state_OTH', 'conn_state_REJ', 'conn_state_RSTOS0', 'conn_state_S0', 'conn_state_SF', 'conn_state_RSTR', 'conn_state_S1']

Sequence 1 | label: BENIGN
Shape: torch.Size([32, 17])
Stats — min: -3.355 max: 6.899 mean: 0.132

First 3 timesteps (rows):
  t=0: [ 0.258 -0.783 -0.193 -0.16  -0.145 -0.195 -0.15   0.     0.     1.
  0.     0.     0.     1.     0.     0.     0.   ]
  t=1: [ 0.258 -0.91  -0.193 -0.16  -0.145 -0.195 -0.15   0.     0.     1.
  0.

In [6]:
class LSTMEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, num_layers=2):
        super().__init__()
        self.lstm    = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers,
                               batch_first=True, dropout=0.2)
        self.mu      = nn.Linear(hidden_dim, latent_dim)
        self.log_var = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        h = h_n[-1]
        return self.mu(h), self.log_var(h)


class LSTMDecoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, num_layers=2):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.z_to_h     = nn.Linear(latent_dim, hidden_dim * num_layers)
        self.z_to_c     = nn.Linear(latent_dim, hidden_dim * num_layers)
        self.lstm        = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers,
                                   batch_first=True, dropout=0.2)
        self.out         = nn.Linear(hidden_dim, input_dim)

    def forward(self, z, input_dim):
        B  = z.size(0)
        h0 = self.z_to_h(z).view(self.num_layers, B, self.hidden_dim)
        c0 = self.z_to_c(z).view(self.num_layers, B, self.hidden_dim)
        x_in    = torch.zeros(B, SEQ_LEN, input_dim).to(z.device)
        out, _  = self.lstm(x_in, (h0, c0))
        return self.out(out)


class RVAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super().__init__()
        self.input_dim = input_dim
        self.encoder   = LSTMEncoder(input_dim, hidden_dim, latent_dim)
        self.decoder   = LSTMDecoder(input_dim, hidden_dim, latent_dim)

    def reparameterise(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        return mu + std * torch.randn_like(std)

    def forward(self, x):
        mu, log_var = self.encoder(x)
        z           = self.reparameterise(mu, log_var)
        return self.decoder(z, self.input_dim), mu, log_var


def vae_loss(recon, x, mu, log_var):
    recon_loss = F.mse_loss(recon, x, reduction='sum') / x.size(0)
    kl_loss    = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / x.size(0)
    return recon_loss + LMBDA * kl_loss, recon_loss, kl_loss


In [7]:
def train(path, scaler, feature_cols):
    input_dim = len(feature_cols)
    model     = RVAE(input_dim, HIDDEN_DIM, LATENT_DIM).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    for epoch in range(EPOCHS):
        model.train()
        dataset = ParquetSequenceDataset(path, scaler, feature_cols, benign_only=True)
        loader  = DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=0)

        total_loss = total_recon = total_kl = n = 0
        for x in loader:
            x = x.to(DEVICE)
            recon, mu, log_var  = model(x)
            loss, recon_l, kl_l = vae_loss(recon, x, mu, log_var)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            b            = x.size(0)
            total_loss  += loss.item()    * b
            total_recon += recon_l.item() * b
            total_kl    += kl_l.item()    * b
            n           += b

        scheduler.step()
        print(f"Epoch {epoch+1}/{EPOCHS} | "
              f"loss {total_loss/n:.4f} | "
              f"recon {total_recon/n:.4f} | "
              f"kl {total_kl/n:.4f} | "
              f"sequences {n:,}")

    return model


In [8]:
def compute_threshold(path, model, scaler, feature_cols):
    model.eval()
    errors  = []
    dataset = ParquetSequenceDataset(path, scaler, feature_cols, benign_only=True)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=0)

    with torch.no_grad():
        for x in loader:
            x = x.to(DEVICE)
            recon, _, _ = model(x)
            errs = F.mse_loss(recon, x, reduction='none').mean(dim=[1, 2]).cpu().numpy()
            errors.append(errs)

    errors    = np.concatenate(errors)
    threshold = np.percentile(errors, 95)
    print(f"Threshold (95th pct benign): {threshold:.4f}")
    return threshold

In [9]:
def evaluate(path, model, scaler, feature_cols, threshold):
    model.eval()
    all_errors = []
    all_labels = []
    dataset    = ParquetSequenceDataset(path, scaler, feature_cols,
                                        benign_only=False, yield_labels=True)
    loader     = DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=0)

    with torch.no_grad():
        for x, lbl in loader:
            x    = x.to(DEVICE)
            recon, _, _ = model(x)
            errs = F.mse_loss(recon, x, reduction='none').mean(dim=[1, 2]).cpu().numpy()
            all_errors.append(errs)
            all_labels.append(lbl.numpy())

    errors = np.concatenate(all_errors)
    labels = np.concatenate(all_labels)

    precisions, recalls, thresholds = precision_recall_curve(labels, errors)
    f1_scores   = 2 * precisions * recalls / (precisions + recalls + 1e-8)
    best_thresh = thresholds[np.argmax(f1_scores[:-1])]

    for name, thresh in [("95th pct", threshold), ("best F1", best_thresh)]:
        preds = (errors > thresh).astype(int)
        print(f"\n── {name} (thresh={thresh:.4f}) ──")
        print(classification_report(labels, preds, target_names=['Benign', 'Malicious']))

    print(f"AUROC: {roc_auc_score(labels, errors):.4f}")
    return errors, labels

Features after encoding: 15
Benign sample rows for scaler: 13,332


Training: 1370it [01:21, 16.74it/s, loss=366193477276.0676, recon=366193477275.3411, kl=1.4580, seqs=43840/500] 


KeyboardInterrupt: 